# Chạy một phương pháp so sánh (Kaggle, GPU T4, bật Internet)

Đổi `METHOD` để chọn phương pháp (`methods/<METHOD>/` trong repo `baselines`), `RUNS` để chọn dataset.
Chỉ clone repo `baselines`: mã nguồn các phương pháp đã chép sẵn trong `third_party/`, không clone repo đồ án, không cài môi trường MonoGS.
Mỗi run cho **một file zip** trong `/kaggle/working/`. Tải zip về rồi chấm cục bộ:
`python baselines/common/score_zip.py --zip <zip> --dataset <dataset> --no_wandb`.

Output của cài đặt và của từng run **hiện ngay trong notebook** (các dòng tiến trình lặp như thanh tqdm được rút bớt, lỗi luôn hiện)
và đầy đủ trong file log. Mỗi bước bị dừng khi quá giờ, hoặc khi **không in gì** trong `STALL_MIN` phút (treo).

Mỗi phiên notebook chỉ chạy **một** `METHOD`: hai phương pháp có thể cài các gói trùng tên (rasterizer của SplaTAM và GS-ICP).

In [ ]:
METHOD = "splatam"  # một thư mục trong methods/ có adapter.py và setup_kaggle.sh: "splatam", "gsicp", "orbslam3", "photoslam", "loopsplat"

OFFICE0_LIT="/kaggle/input/replica-lunogs/office0/lit"
OFFICE0_EVEN="/kaggle/input/replica-lunogs/office0/even"
OFFICE0_AMB="/kaggle/input/datasets/nguyntrungan/office0"

ROOM0_LIT="/kaggle/input/datasets/nguyntrungan/replica-lunogs/room0/lit"
ROOM0_EVEN="/kaggle/input/datasets/nguyntrungan/replica-lunogs/room0/even"

RUNS = [OFFICE0_LIT, OFFICE0_EVEN]  # chạy tuần tự, theo thứ tự này
# Frame giữa hai checkpoint, để chạy tiếp khi phiên bị ngắt. 0 = tắt; bắt buộc 0 với phương pháp
# không có checkpoint (gsicp, orbslam3, photoslam, loopsplat), adapter của nó từ chối giá trị khác.
CKPT_EVERY = {"splatam": 100}.get(METHOD, 0)
NO_MAP = False    # True: không đưa bản đồ cuối vào zip (zip nhỏ hơn, không render lại được)

# Giới hạn thời gian. Phiên Kaggle tối đa 12 giờ cho TẤT CẢ các bước.
SETUP_TIMEOUT_MIN = 40  # cài đặt + build extension
TIMEOUT_H = 5.5         # mỗi run; hết giờ thì dừng, chạy tiếp được từ checkpoint
STALL_MIN = 20          # dừng một bước nếu nó không in gì trong chừng này phút (treo)
PROGRESS_S = 30         # in tối đa một dòng tiến trình lặp mỗi chừng này giây

OUT = "/kaggle/working"
CODE = "/tmp/code"  # mã nguồn ngoài /kaggle/working: output chỉ giữ các zip

In [ ]:
%%time
import sys
!rm -rf {CODE}/baselines && mkdir -p {CODE} && git clone -q --depth 1 https://github.com/Trung4n/baselines {CODE}/baselines && git -C {CODE}/baselines log -1 --format='baselines %H %s'
sys.path.insert(0, f"{CODE}/baselines/common")
import kaggle_runner as K

# methods/<METHOD>/setup_kaggle.sh: cài phụ thuộc bằng uv, build extension cho GPU này. Lỗi hoặc treo thì dừng ở đây.
K.setup(METHOD, SETUP_TIMEOUT_MIN, STALL_MIN, PROGRESS_S)

Log đầy đủ của mỗi run ở `<OUT>/<METHOD>__<scene>_<variant>/run.log` và nằm trong zip. Các run chạy **tuần tự**: FPS phải đo khi máy không bị chia sẻ.
Run lỗi, quá giờ hoặc treo trả về `None`; các run sau vẫn chạy.

In [ ]:
zips = [K.run(METHOD, d, OUT, CKPT_EVERY, TIMEOUT_H, STALL_MIN, PROGRESS_S, no_map=NO_MAP) for d in RUNS]
zips

**Chạy tiếp sau khi phiên bị ngắt hoặc run bị dừng:** nếu là phiên mới, thêm output của phiên trước làm input và chép thư mục
`<METHOD>__<scene>_<variant>` của run dở về `/kaggle/working`; rồi chạy ô dưới.
Frame của checkpoint được xử lý lại một lần; `timing.json` cộng các phiên và ghi rõ là đã resume.

In [ ]:
# !cp -r /kaggle/input/<output phiên trước>/splatam__office0_lit {OUT}/
# K.run(METHOD, OFFICE0_LIT, OUT, CKPT_EVERY, TIMEOUT_H, STALL_MIN, PROGRESS_S, resume=True, no_map=NO_MAP)

In [ ]:
# Kiểm mọi zip của METHOD rồi xóa thư mục làm việc của run đã đóng gói: output chỉ còn các zip để tải về.
K.collect(METHOD, OUT)